# API Design (FastAPI + Pydantic)

## Agenda (high-level)
1. Expectations + objectives
2. Recap: OOP pipeline → service layer → API boundary
3. Environment setup (venv, dependencies, running FastAPI)
4. Guided build: `POST /normalize-text`
5. Validation + API contract behavior (422)
6. Testing basics: pytest + TestClient
7. Lab: `POST /keywords-simple` + validations + tests
8. Wrap-up + next session preview

## Session expectations
- You will **run commands in a terminal** (venv, uvicorn, pytest).
- We will keep endpoints **deterministic** (no LLM yet).

## Session objectives
By the end of this session:
1. Explain the role of an API as a “data contract boundary”.
2. Create and run a basic FastAPI service locally.
3. Define request/response schemas using Pydantic.
4. Separate route handlers from service logic (OOP layering).
5. Write simple endpoint tests with pytest + TestClient.
6. Complete a lab extending endpoints and validations.

## 2) Quick recap: OOP pipeline → service layer → API boundary (10–15 min)


## 3) Environment setup

### 3.1 Create and activate a virtual environment (venv)

Run these in a terminal in your project folder:

**macOS / Linux**
```bash
python -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
```

**Windows (PowerShell)**
```powershell
python -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install --upgrade pip
```

### 3.2 Install dependencies
We’ll use only: `fastapi`, `uvicorn`, `pydantic`, `pytest`, `httpx`.

### 3.3 Run FastAPI
From the same folder where `app.py` exists:
```bash
uvicorn app:app --reload
```

**What does `--reload` do?**
- It watches your code files; when you save changes, it restarts the server automatically.
- It’s great for development, but it’s not used for production deployments.

### 3.4 `requirements.txt`
- It’s a minimal “dependency contract” for your project.
- In teams, this is how you keep environments reproducible.

### Common mistakes / troubleshooting
- **Port in use**: change port: `uvicorn app:app --reload --port 8001`
- **venv not activated**: `which python` (mac/linux) or `where python` (windows)
- **pytest not found**: run `python -m pytest -q` instead of `pytest -q`


In [ ]:
# Create a minimal project on disk: app.py, requirements.txt, tests/test_api.py

from pathlib import Path

project_root = Path.cwd()
tests_dir = project_root / "tests"
tests_dir.mkdir(parents=True, exist_ok=True)

requirements_txt = """fastapi
uvicorn
pydantic
pytest
httpx
"""

(project_root / "requirements.txt").write_text(requirements_txt, encoding="utf-8")

print("Wrote requirements.txt")
print((project_root / "requirements.txt").read_text(encoding="utf-8"))


## 4) “Hello API” guided build

### Suggested project structure (what we would do outside a notebook)
- `app.py` (FastAPI application + routes)
- `schemas/` (Pydantic request/response models)
- `services/` (business logic / transformations)
- `tests/` (pytest tests)

In this notebook we will **simulate** that structure but keep everything runnable by writing a single `app.py`.

### Endpoint #1: `POST /normalize-text`
- Input schema: `NormalizeTextRequest(text: str, lowercase: bool=False)`
- Output schema: `NormalizeTextResponse(normalized_text: str, char_count: int, word_count: int)`

In [ ]:
# Add project_root to app.py below
print(f"{project_root}/app.py")

In [ ]:
%%writefile app.py

import re
from typing import Tuple

from fastapi import FastAPI
from pydantic import BaseModel, Field


app = FastAPI(title="Module 1 - Session 1 API", version="1.0.0")


# -----------------------------
# Schemas (Pydantic models)
# -----------------------------

class NormalizeTextRequest(BaseModel):
    text: str
    lowercase: bool = False


class NormalizeTextResponse(BaseModel):
    normalized_text: str
    char_count: int
    word_count: int


# -----------------------------
# Services (business logic)
# -----------------------------

def _collapse_whitespace(value: str) -> str:
    return re.sub(r"\s+", " ", value).strip()


def normalize_text(text: str, lowercase: bool = False) -> Tuple[str, int, int]:
    """Deterministic text normalization.

    Design notes:
    - Keep this function pure-ish (no global state, no I/O).
    - This style is easy to test and safe to reuse in pipelines.
    """
    if lowercase:
        text = text.lower()

    normalized = _collapse_whitespace(text)
    char_count = len(normalized)
    word_count = 0 if not normalized else len(normalized.split(" "))
    return normalized, char_count, word_count


# -----------------------------
# Routes (API boundary)
# -----------------------------

@app.post("/normalize-text", response_model=NormalizeTextResponse)
def post_normalize_text(payload: NormalizeTextRequest) -> NormalizeTextResponse:
    normalized, char_count, word_count = normalize_text(
        text=payload.text,
        lowercase=payload.lowercase,
    )
    return NormalizeTextResponse(
        normalized_text=normalized,
        char_count=char_count,
        word_count=word_count,
    )

In [ ]:
print("app.py size:", (project_root / "app.py").stat().st_size, "bytes")

### How to run the API (terminal)

From the folder that contains `app.py`:
```bash
uvicorn app:app --reload
```

Then open:
- Swagger UI: `http://127.0.0.1:8000/docs`
- ReDoc UI: `http://127.0.0.1:8000/redoc`
- OpenAPI JSON: `http://127.0.0.1:8000/openapi.json`

**Why this matters:**
- FastAPI generates docs from your schemas. This meand our API is now self-describing! :D


In [ ]:
# Quick in-notebook smoke test (no server needed): use TestClient to call the app directly.

from fastapi.testclient import TestClient
import importlib

app_module = importlib.import_module("app")
client = TestClient(app_module.app)

payload = {
    "text": "  Hello Python Academy Trainees!   ",
    "lowercase": False
}

resp = client.post("/normalize-text", json=payload)
print("Status:", resp.status_code)
print(resp.json())


## 5) API contract and validation

Now we’ll add a schema constraint:
- `text` must be at least **20 characters**.

Rememner our last sessions and validation/error handling:
- Upstream producers often send incomplete data.
- If you validate early, you avoid downstream processing of low-quality records.

FastAPI behavior:
- If the request body doesn’t match the schema, FastAPI returns **422 Unprocessable Entity**.
- This is a *contract-level* failure, not a business-rule failure.

### Sample requests/responses:

**Request (valid)**
```json
{
  "text": "Hello Python Academy Trainees!",
  "lowercase": true
}
```

**Response (200)**
```json
{
  "normalized_text": "hello python academy trainees!",
  "char_count": 30,
  "word_count": 4
}
```

**Request (invalid: too short)**
```json
{
  "text": "Too short",
  "lowercase": false
}
```

**Response (422)**
```json
{
  "detail": [
    {
      "loc": ["body", "text"],
      "msg": "String should have at least 20 characters",
      "type": "string_too_short"
    }
  ]
}
```


In [ ]:
%%writefile ____
# Add the correct path for the file!
# Update app.py to enforce: text min length = 20 # App with Validation

import re
from typing import Tuple

from fastapi import FastAPI
from pydantic import BaseModel, Field


app = FastAPI(title="Module 1 - Session 1 API", version="1.0.0")


# -----------------------------
# Schemas (Pydantic models)
# -----------------------------

class NormalizeTextRequest(BaseModel):
    # Contract-level constraint: reject short inputs early
    text: str = Field(..., min_length=20)
    lowercase: bool = False


class NormalizeTextResponse(BaseModel):
    normalized_text: str
    char_count: int
    word_count: int


# -----------------------------
# Services (business logic)
# -----------------------------

def _collapse_whitespace(value: str) -> str:
    return re.sub(r"\s+", " ", value).strip()


def normalize_text(text: str, lowercase: bool = False) -> Tuple[str, int, int]:
    """Deterministic text normalization."""
    if lowercase:
        text = text.lower()

    normalized = _collapse_whitespace(text)
    char_count = len(normalized)
    word_count = 0 if not normalized else len(normalized.split(" "))
    return normalized, char_count, word_count


# -----------------------------
# Routes (API boundary)
# -----------------------------

@app.post("/normalize-text", response_model=NormalizeTextResponse)
def post_normalize_text(payload: NormalizeTextRequest) -> NormalizeTextResponse:
    normalized, char_count, word_count = normalize_text(
        text=payload.text,
        lowercase=payload.lowercase,
    )
    return NormalizeTextResponse(
        normalized_text=normalized,
        char_count=char_count,
        word_count=word_count,
    )

In [ ]:
# Reload module to reflect changes
import importlib
import app as app_module
importlib.reload(app_module)

from fastapi.testclient import TestClient
client = TestClient(app_module.app)

valid = {"text": "Hello Python Academy Trainees!", "lowercase": True}
invalid = {"text": "Too short", "lowercase": False}

r1 = client.post("/normalize-text", json=valid)
r2 = client.post("/normalize-text", json=invalid)

print("Valid status:", r1.status_code)
print("Valid body:", r1.json())
print("Invalid status:", r2.status_code)
print("Invalid body (detail keys):", list(r2.json().keys()))
print("Invalid detail sample:", r2.json().get("detail", [])[:1])

## 6) Testing basics (pytest)

Testing an API:
- You’re testing the **contract** and the **deterministic behavior**.
- If your downstream pipeline calls this service, tests prevent “silent changes”.

We’ll add 3 tests:
1. Happy path returns 200 + expected fields
2. Invalid payload returns 422
3. `lowercase=true` affects output

### How to run (terminal)
```bash
pytest -q
```
If that fails because `pytest` is not in PATH:
```bash
python -m pytest -q
```

### Minimal `tests/test_api.py` content
```python
import pytest
from fastapi.testclient import TestClient
import app

client = TestClient(app.app)

def test_normalize_text_happy_path():
    payload = {"text": "Hello Data Engineering world!", "lowercase": False}
    resp = client.post("/normalize-text", json=payload)
    assert resp.status_code == 200
    data = resp.json()
    assert "normalized_text" in data
    assert "char_count" in data
    assert "word_count" in data

def test_normalize_text_invalid_payload_returns_422():
    payload = {"text": "Too short", "lowercase": False}
    resp = client.post("/normalize-text", json=payload)
    assert resp.status_code == 422

def test_normalize_text_lowercase_changes_output():
    payload = {"text": "Hello Data Engineering world!", "lowercase": True}
    resp = client.post("/normalize-text", json=payload)
    assert resp.status_code == 200
    assert resp.json()["normalized_text"].startswith("hello")
```


In [ ]:
%%writefile ____
# Write tests/test_api.py to disk

import app
from fastapi.testclient import TestClient

client = TestClient(app.app)


def test_normalize_text_happy_path():
    payload = {"text": "Hello Data Engineering world!", "lowercase": False}
    resp = client.post("/normalize-text", json=payload)
    assert resp.status_code == 200
    data = resp.json()
    assert "normalized_text" in data
    assert "char_count" in data
    assert "word_count" in data
    assert isinstance(data["char_count"], int)
    assert isinstance(data["word_count"], int)


def test_normalize_text_invalid_payload_returns_422():
    payload = {"text": "Too short", "lowercase": False}
    resp = client.post("/normalize-text", json=payload)
    assert resp.status_code == 422


def test_normalize_text_lowercase_changes_output():
    payload = {"text": "Hello Data Engineering world!", "lowercase": True}
    resp = client.post("/normalize-text", json=payload)
    assert resp.status_code == 200
    assert resp.json()["normalized_text"].startswith("hello")

In [ ]:
# Run pytest from within the notebook (still mirrors the terminal command) or use !

import sys
import subprocess

result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    capture_output=True,
    text=True,
)

print("pytest exit code:", result.returncode)
print("--- stdout ---")
print(result.stdout)
print("--- stderr ---")
print(result.stderr)

assert result.returncode == 0, "Pytest failed. Read stdout/stderr above."


## 7) Lab (30–40 min)

You will extend the API with a second endpoint and add business-rule validation + tests.

### Task A: Add endpoint `POST /keywords-simple`

**Input schema**
- `KeywordsRequest(text: str, top_k: int=5)`

**Output schema**
- `KeywordsResponse(keywords: list[str])`

**Baseline keyword extraction (deterministic)**
1. Lowercase
2. Remove punctuation
3. Split on whitespace
4. Count frequency
5. Return top `top_k` tokens

> Data Engineering mindset: this is a tiny “feature extraction” stage.

### Task B: Add business-rule validation
- If `top_k < 1` or `top_k > 20`, raise `HTTPException(400)` with a clear message.

**Important:** This is different from schema validation (422).
- 422 = request body does not satisfy the contract schema.
- 400 = request is well-formed but violates a business rule.

### Task C: Add tests
- Happy path: returns 200 + list of keywords size ≤ top_k
- Invalid `top_k` (0 or 21): returns 400

### Student placeholders
Below we write a **starter template** to `lab_starter.txt` so you can copy/paste into `app.py` and `tests/test_api.py`.
You do not need to run anything in this section until you implement the endpoint.

Common mistakes
- Forgetting to import `HTTPException`.
- Returning a plain dict instead of a Pydantic response model.
- Using non-deterministic ordering (fix by sorting ties consistently).


In [ ]:
%%writefile lab_starter.txt

# Write lab starter guidance to disk (does not affect the running code yet)

r"""LAB STARTER (copy/paste hints)

1) In app.py:
- Add Pydantic models: KeywordsRequest, KeywordsResponse
- Add a service function: extract_keywords_simple(text: str, top_k: int) -> list[str]
- Add route: POST /keywords-simple
- Add business-rule validation for top_k using HTTPException(400)

2) In tests/test_api.py:
- Add two tests:
  a) POST /keywords-simple happy path
  b) POST /keywords-simple top_k invalid returns 400

Suggested happy-path payload:
{
  "text": "Data Engineering is engineering data. Data quality matters a lot.",
  "top_k": 3
}

Expected response shape:
{
  "keywords": ["data", "engineering", "..."]
}
"""

In [ ]:
# Reload app module and run pytest again (solution should pass)

import importlib
import app as app_module
importlib.reload(app_module)

import sys
import subprocess

result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    capture_output=True,
    text=True,
)

print("pytest exit code:", result.returncode)
print("--- stdout ---")
print(result.stdout)
print("--- stderr ---")
print(result.stderr)

assert result.returncode == 0, "Pytest failed. Read stdout/stderr above."


## Wrap-up

### Key takeaways
- An API is a **data contract boundary**: it stabilizes how upstream and downstream systems interact.
- Pydantic schemas give you **typed contracts**, **validation**, and **auto-generated docs**.
- Keep route handlers **thin**; move deterministic logic to **services**.
- FastAPI distinguishes **schema validation errors (422)** from **business rule errors (400)**.
- Tests protect the contract: status codes + response shape + critical behavior.

### Next session
- Error handling patterns
- Logging and debug visibility
- Configuration (env vars, settings)
- Stronger project layout for multi-module services


> Content created by [**Carlos Cruz-Maldonado**](https://www.linkedin.com/in/carloscruzmaldonado/).  
> I am available to answer any questions or provide further assistance.   
> Feel free to reach out to me at any time.

In [1]:
%pip install fastapi

In [2]:
%pip install uvicorn

In [3]:
%%writefile firstAPI.py
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

@app.get("/")
def root():
  return "home page"


Writing firstAPI.py


In [9]:
uvicorn app:firstAPI --reload

SyntaxError: invalid syntax (2451809437.py, line 1)

In [13]:
!uvicorn firstAPI:app --reload

INFO:     Will watch for changes in these directories: ['/content']
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [5382] using WatchFiles
INFO:     Started server process [5384]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [5384]
INFO:     Stopping reloader process [5382]


In [21]:
from fastapi.testclient import TestClient
import importlib

fastAPIapp = importlib.import_module("firstAPI")

client = TestClient(fastAPIapp.app)

resp = client.get("/")
print("Status:", resp.status_code,resp.json())


Status: 200 home page


# Task
Create the `secondAPI.py` file including the FastAPI app instance and necessary imports.

## Create secondAPI.py with basic FastAPI setup

### Subtask:
Generate the `secondAPI.py` file including the FastAPI app instance and necessary imports.


**Reasoning**:
To generate the `secondAPI.py` file with the basic FastAPI setup as requested, I will use the `%%writefile` magic command to create the file and populate it with the necessary import and app instantiation.



In [1]:
%%writefile secondAPI.py
from fastapi import FastAPI

app = FastAPI()

Writing secondAPI.py


## Define Pydantic Models for Keywords

### Subtask:
Add the `KeywordsRequest` and `KeywordsResponse` Pydantic models to `secondAPI.py`.


**Reasoning**:
To define the Pydantic models as requested, I will append the necessary imports and model definitions to the `secondAPI.py` file using the `%%writefile -a` magic command.



In [2]:
%%writefile -a secondAPI.py

from pydantic import BaseModel

class KeywordsRequest(BaseModel):
    text: str
    top_k: int = 5


class KeywordsResponse(BaseModel):
    keywords: list[str]

Appending to secondAPI.py


## Implement Keyword Extraction Logic

### Subtask:
Create the `extract_keywords_simple` service function within `secondAPI.py` to handle the keyword extraction steps (lowercase, remove punctuation, split, count frequency, return top_k).


**Reasoning**:
To implement the keyword extraction logic, I need to append the necessary imports, a helper function for punctuation removal, and the main `extract_keywords_simple` function to `secondAPI.py`.



In [3]:
%%writefile -a secondAPI.py

import re
from collections import Counter


def _remove_punctuation(text: str) -> str:
    """Removes punctuation from the given text."""
    return re.sub(r'[^À-ſ\w\s]', '', text)

def extract_keywords_simple(text: str, top_k: int) -> list[str]:
    """Extracts top_k keywords from text based on frequency, handling ties deterministically."""
    # 1. Lowercase
    text = text.lower()

    # 2. Remove punctuation
    cleaned_text = _remove_punctuation(text)

    # 3. Split on whitespace
    words = cleaned_text.split()

    # Handle empty text case
    if not words:
        return []

    # 4. Count frequency
    word_counts = Counter(words)

    # 5. Return top top_k tokens, sorting ties alphabetically
    # Get all items and sort them: first by count (desc), then by word (asc)
    sorted_word_counts = sorted(word_counts.items(), key=lambda item: (-item[1], item[0]))

    keywords = [word for word, count in sorted_word_counts[:top_k]]
    return keywords

Appending to secondAPI.py


## Add Keywords API Endpoint

### Subtask:
Implement the `POST /keywords-simple` route in `secondAPI.py`, integrating the Pydantic models, the `extract_keywords_simple` service, and the business rule validation for `top_k` using `HTTPException(400)`.


**Reasoning**:
To implement the `POST /keywords-simple` route with the specified business logic and validation, I need to update the `secondAPI.py` file. This involves adding `HTTPException` to the imports, defining the new route, implementing the `top_k` validation, calling the `extract_keywords_simple` service function, and returning the `KeywordsResponse`. I will reconstruct the entire `secondAPI.py` content to ensure correct imports and structure.



In [4]:
%%writefile secondAPI.py

import re
from collections import Counter
from typing import List

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field


app = FastAPI()


# -----------------------------
# Schemas (Pydantic models)
# -----------------------------

class KeywordsRequest(BaseModel):
    text: str
    top_k: int = 5


class KeywordsResponse(BaseModel):
    keywords: List[str]


# -----------------------------
# Services (business logic)
# -----------------------------

def _remove_punctuation(text: str) -> str:
    """Removes punctuation from the given text."""
    return re.sub(r'[^À-ſ\w\s]', '', text)

def extract_keywords_simple(text: str, top_k: int) -> List[str]:
    """Extracts top_k keywords from text based on frequency, handling ties deterministically."""
    # 1. Lowercase
    text = text.lower()

    # 2. Remove punctuation
    cleaned_text = _remove_punctuation(text)

    # 3. Split on whitespace
    words = cleaned_text.split()

    # Handle empty text case
    if not words:
        return []

    # 4. Count frequency
    word_counts = Counter(words)

    # 5. Return top top_k tokens, sorting ties alphabetically
    # Get all items and sort them: first by count (desc), then by word (asc)
    sorted_word_counts = sorted(word_counts.items(), key=lambda item: (-item[1], item[0]))

    keywords = [word for word, count in sorted_word_counts[:top_k]]
    return keywords


# -----------------------------
# Routes (API boundary)
# -----------------------------

@app.post("/keywords-simple", response_model=KeywordsResponse)
def post_keywords_simple(payload: KeywordsRequest) -> KeywordsResponse:
    # Business rule validation for top_k
    if not (1 <= payload.top_k <= 20):
        raise HTTPException(status_code=400, detail="top_k must be between 1 and 20 (inclusive).")

    keywords = extract_keywords_simple(text=payload.text, top_k=payload.top_k)
    return KeywordsResponse(keywords=keywords)

Overwriting secondAPI.py


In [5]:
!uvicorn secondAPI:app --reload

INFO:     Will watch for changes in these directories: ['/content']
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [4976] using WatchFiles
INFO:     Started server process [4978]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [4978]
INFO:     Stopping reloader process [4976]


In [7]:
from fastapi.testclient import TestClient
import importlib

# Load the secondAPI module
secondAPI_module = importlib.import_module("secondAPI")
client = TestClient(secondAPI_module.app)

print("--- Valid Request Example ---")
# Example with valid input
valid_payload = {
    "text": "Data Engineering is engineering data. Data quality matters a lot.",
    "top_k": 3
}

resp_valid = client.post("/keywords-simple", json=valid_payload)
print("Status Code:", resp_valid.status_code)
print("Response Body:", resp_valid.json())

print("\n--- Invalid Request Example (top_k out of range) ---")
# Example with invalid top_k (business rule validation)
invalid_payload_top_k = {
    "text": "This is some sample text for testing.",
    "top_k": 0 # Invalid top_k, must be between 1 and 20
}

resp_invalid_top_k = client.post("/keywords-simple", json=invalid_payload_top_k)
print("Status Code:", resp_invalid_top_k.status_code)
print("Response Body:", resp_invalid_top_k.json())

print("\n--- Invalid Request Example (missing required 'text' field) ---")
# Example with invalid payload (schema validation)
invalid_payload_schema = {
    "top_k": 5
} # 'text' field is missing

resp_invalid_schema = client.post("/keywords-simple", json=invalid_payload_schema)
print("Status Code:", resp_invalid_schema.status_code)
print("Response Body:", resp_invalid_schema.json())


--- Valid Request Example ---
Status Code: 200
Response Body: {'keywords': ['data', 'engineering', 'a']}

--- Invalid Request Example (top_k out of range) ---
Status Code: 400
Response Body: {'detail': 'top_k must be between 1 and 20 (inclusive).'}

--- Invalid Request Example (missing required 'text' field) ---
Status Code: 422
Response Body: {'detail': [{'type': 'missing', 'loc': ['body', 'text'], 'msg': 'Field required', 'input': {'top_k': 5}}]}


## Final Task

### Subtask:
Verify that `secondAPI.py` contains all the requested components and functionality.


## Summary:

### Q&A
The `secondAPI.py` file now contains all the requested components and functionality, including the FastAPI app instance, necessary imports, Pydantic models for request and response, a keyword extraction service function, and a `POST /keywords-simple` API endpoint with input validation.

### Data Analysis Key Findings
*   The `secondAPI.py` file was successfully created and populated with a basic FastAPI application setup, including the instantiation of `app = FastAPI()`.
*   Pydantic models `KeywordsRequest` (with `text: str` and `top_k: int = 5`) and `KeywordsResponse` (with `keywords: list[str]`) were defined to handle API request and response data.
*   A `_remove_punctuation` helper function was implemented using regular expressions (`re.sub(r'[^À-ſ\w\s]', '', text)`) to ensure accurate text cleaning, including Unicode characters.
*   The `extract_keywords_simple` service function was developed to perform keyword extraction by lowercasing text, removing punctuation, splitting words, counting frequencies with `collections.Counter`, and deterministically sorting words first by count (descending) and then alphabetically (ascending) to retrieve the top `top_k` keywords.
*   A `POST /keywords-simple` API endpoint was implemented, integrating the `KeywordsRequest` and `KeywordsResponse` models. It includes a business rule validation to ensure `top_k` is between 1 and 20 (inclusive), raising an `HTTPException(400)` for invalid values.

### Insights or Next Steps
*   The API is ready for deployment and can be tested using standard API testing tools (e.g., cURL, Postman) to verify its functionality and adherence to the defined business rules.
*   Consider implementing more advanced keyword extraction techniques (e.g., TF-IDF, spaCy, NLTK) to improve the quality and relevance of extracted keywords for more complex text inputs.


# Task
Create the `tests` directory, create the `tests/test_secondAPI.py` file with test cases for the `POST /keywords-simple` endpoint (including valid input, invalid `top_k`, and missing `text`), and then run pytest to verify the tests.

## Ensure tests directory exists

### Subtask:
Create the 'tests' directory if it doesn't already exist, to house our test files.


**Reasoning**:
To create the 'tests' directory as instructed, I will use the `pathlib` module to define the directory path and then use `mkdir` with `parents=True` and `exist_ok=True` to create it idempotently.



In [8]:
from pathlib import Path

# Define the path for the 'tests' directory
tests_dir = Path.cwd() / "tests"

# Create the 'tests' directory if it doesn't already exist
tests_dir.mkdir(parents=True, exist_ok=True)

print(f"Ensured directory '{tests_dir}' exists.")

Ensured directory '/content/tests' exists.


## Create `tests/test_secondAPI.py` with Test Cases

### Subtask:
Create the `tests/test_secondAPI.py` file containing test cases for the `POST /keywords-simple` endpoint. These tests will include:
- **Happy path**: Verify a 200 status code and the expected format of keywords.
- **Invalid `top_k`**: Confirm a 400 status code for out-of-range `top_k` values.
- **Missing `text` field**: Assert a 422 status code for schema validation errors.

### Reasoning:
To create the `tests/test_secondAPI.py` file with the specified test cases, I will use the `%%writefile` magic command. The tests will utilize FastAPI's `TestClient` to interact with the `secondAPI` application and `pytest` assertions to validate responses.

**Reasoning**:
Now that the 'tests' directory is ensured to exist, I will create the `tests/test_secondAPI.py` file. This file will contain test cases for the `POST /keywords-simple` endpoint, covering happy path, invalid `top_k`, and missing `text` scenarios, as specified in the main task.



In [11]:
%%writefile test_secondAPI.py

from fastapi.testclient import TestClient
import pytest
import importlib

# Load the secondAPI module dynamically
secondAPI_module = importlib.import_module("secondAPI")
client = TestClient(secondAPI_module.app)

def test_keywords_simple_happy_path():
    payload = {
        "text": "Data Engineering is engineering data. Data quality matters a lot.",
        "top_k": 3
    }
    resp = client.post("/keywords-simple", json=payload)
    assert resp.status_code == 200
    data = resp.json()
    assert "keywords" in data
    assert isinstance(data["keywords"], list)
    assert len(data["keywords"]) <= payload["top_k"]
    assert data["keywords"] == ["data", "engineering", "a"] # Deterministic check

def test_keywords_simple_invalid_top_k_returns_400():
    payload = {
        "text": "This is some sample text for testing.",
        "top_k": 0 # Invalid top_k
    }
    resp = client.post("/keywords-simple", json=payload)
    assert resp.status_code == 400
    assert resp.json() == {"detail": "top_k must be between 1 and 20 (inclusive)."}

    payload["top_k"] = 21 # Another invalid top_k
    resp = client.post("/keywords-simple", json=payload)
    assert resp.status_code == 400
    assert resp.json() == {"detail": "top_k must be between 1 and 20 (inclusive)."}

def test_keywords_simple_missing_text_returns_422():
    payload = {
        "top_k": 5
    } # Missing 'text' field
    resp = client.post("/keywords-simple", json=payload)
    assert resp.status_code == 422
    assert "detail" in resp.json()
    assert any(item["loc"] == ["body", "text"] and "Field required" in item["msg"] for item in resp.json()["detail"])


Overwriting test_secondAPI.py


In [14]:
!python -m pytest -q -v

============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-8.4.2, pluggy-1.6.0
rootdir: /content
plugins: langsmith-0.7.6, anyio-4.12.1, typeguard-4.5.1
collected 3 items / 1 error                                                    

==================================== ERRORS ====================================
___________________ ERROR collecting tests/test_secondAPI.py ___________________
import file mismatch:
imported module 'test_secondAPI' has this __file__ attribute:
  /content/test_secondAPI.py
which is not the same as the test file we want to collect:
  /content/tests/test_secondAPI.py
HINT: remove __pycache__ / .pyc files and/or use a unique basename for your test file modules
=========================== short test summary info ============================
ERROR tests/test_secondAPI.py
!!!!!!!!!!!!!!!!!!!! Interrupted: 1 error during collection !!!!!!!!!!!!!!!!!!!!
=============================== 1 error in 0.41s

In [15]:
%ls -ltr test_secondAPI.py

-rw-r--r-- 1 root root 1640 Feb 26 22:36 test_secondAPI.py


In [16]:
!python -m pytest -q -v test_secondAPI.py

============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-8.4.2, pluggy-1.6.0
rootdir: /content
plugins: langsmith-0.7.6, anyio-4.12.1, typeguard-4.5.1
collected 3 items                                                              

test_secondAPI.py ...                                                    [100%]

============================== 3 passed in 0.28s ===============================
